# Ex-Post Asset Allocation in Iran

## Research notebook

This project studies how gold, Tehran equities, Tehran residential housing, and an exchange-traded fixed-income benchmark behaved across Solar Hijri market regimes. It estimates the hindsight-efficient composition of the risky sleeve for each complete year and separately for the 1405 YTD period through Mordad.

The analysis is deliberately **ex-post**, not a trading recommendation or forecast. Its contribution is a reproducible chain from source-audited monthly levels to returns, asset diagnostics, constrained optimization, and numerical validation.

## Portfolio decision framework

The allocation problem has two distinct stages. Stage I identifies the composition of the risky sleeve; Stage II determines how much total wealth the investor assigns to it. Market history can inform Stage I, but Stage II requires an explicit risk budget or preference.

In **Stage I**, the initial allocation is split across gold, equity, and Tehran housing. Fixed income is the investable benchmark. The risky sleeve is purchased at the beginning of each Solar Hijri year and held without rebalancing, allowing weights to drift.

For initial weights $w$, monthly returns $r_{i,m}$ generate

$$V_m(w)=\sum_{i=1}^{N}w_i\prod_{k=1}^{m}(1+r_{i,k}), \qquad V_0=1,$$

with monthly sleeve return $r_{p,m}(w)=V_m(w)/V_{m-1}(w)-1$ and benchmark differential $d_m(w)=r_{p,m}(w)-r_{b,m}$.

Stage I maximizes the annualized benchmark-relative efficiency ratio

$$w_y^*=\arg\max_w \sqrt{12}\,\frac{\overline{d_y(w)}}{s(d_y(w))},$$

subject to $w_i\ge0$ and $\sum_iw_i=1$. Because the benchmark is an investable ETF rather than a riskless rate, this is interpreted as an information-ratio-style statistic rather than an unqualified Sharpe ratio.

In **Stage II**, the sleeve is combined with fixed income as $w_y^{total}=(\alpha_yw_y^*,1-\alpha_y)$. Rather than claiming one investor-specific answer, the notebook reports $\alpha_y$ over an explicit grid of risk-aversion values.

## Data scope and comparability

The risky universe is gold, the Tehran Stock Exchange total-return index, and Tehran residential housing. Etemad, an exchange-traded fixed-income fund, is the benchmark.

The optimization sample is **1396-1405/05**. Years 1396-1404 contain 12 aligned monthly observations per asset; 1405 is a five-month YTD period through Mordad. Year 1395 is descriptive only because the missing 1394/12 housing level prevents construction of its first monthly return. Housing uses official CBI observations through 1403/05 and a chain-linked Kilid proxy afterward. The 1403 source transition and the Kilid-only 1404-1405 results remain explicitly labelled, but all available years are analyzed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from IPython.display import display
from scipy.optimize import minimize

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [2]:
root_candidates = [Path.cwd(), Path.cwd() / "asset_allocation", Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in root_candidates if (root / "data" / "processed" / "analysis").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the asset_allocation project root from the current working directory."
    )

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "analysis"
RETURNS_PATH = DATA_DIR / "monthly_asset_returns.csv"
LEVELS_PATH = DATA_DIR / "monthly_asset_levels.csv"
HOUSING_AUDIT_PATH = DATA_DIR / "housing_data_quality_audit.csv"

RISKY_ASSETS = ["gold_18k", "equity_tedpix", "tehran_housing"]
FIXED_INCOME = "fixed_income_etemad"
ANALYSIS_YEARS = list(range(1396, 1406))
EXPECTED_MONTHS = {year: (5 if year == 1405 else 12) for year in ANALYSIS_YEARS}
ASSET_LABELS = {"gold_18k": "Gold", "equity_tedpix": "Equity", "tehran_housing": "Housing", "fixed_income_etemad": "Fixed Income"}

for required_path in (RETURNS_PATH, LEVELS_PATH, HOUSING_AUDIT_PATH):
    assert required_path.exists(), f"Missing processed input: {required_path}"

In [3]:
returns_raw = pd.read_csv(RETURNS_PATH)
levels_raw = pd.read_csv(LEVELS_PATH)
housing_audit = pd.read_csv(HOUSING_AUDIT_PATH)

returns_raw["monthly_return"] = pd.to_numeric(returns_raw["monthly_return"], errors="coerce")
levels_raw["month_end_level"] = pd.to_numeric(levels_raw["month_end_level"], errors="coerce")
levels_raw["raw_source_level"] = pd.to_numeric(levels_raw["raw_source_level"], errors="coerce")

assert not returns_raw.duplicated(["jalali_period", "asset_id"]).any()
assert not levels_raw.duplicated(["jalali_period", "asset_id"]).any()
assert np.isfinite(returns_raw["monthly_return"].dropna()).all()
assert levels_raw["month_end_level"].dropna().gt(0).all()
display(returns_raw.head())

,jalali_period,jalali_year,jalali_month,asset_id,previous_jalali_period,previous_month_end_level,current_month_end_level,monthly_return,return_definition,current_source_observation_date,previous_source_observation_date,source_file,data_quality_flag,missing_reason
0,1395/01,1395,1,gold_18k,1394/12,"1,034,900.0000","1,048,700.0000",0.0133,price_appreciation,1395-01-31,1394-12-26,data/raw/gold_18k/tgju_gold_18k_daily.csv,ok,NaN
1,1395/01,1395,1,equity_tedpix,1394/12,"80,219.4000","78,430.9000",-0.0223,total_return_index_level_change,2016-04-19,2016-03-16,data/raw/tse_total_index/tedpix_daily.csv,ok,NaN
2,1395/01,1395,1,fixed_income_etemad,1394/12,"12,583.0000","12,926.0000",0.0273,market_price_return_no_periodic_distribution,2016-04-19,2016-03-16,data/raw/fixed_income/etf/etemad.csv,provisional_distribution_policy_audit,NaN
3,1395/01,1395,1,tehran_housing,1394/12,NaN,42.1870,NaN,transaction_price_appreciation_excludes_rent,1395/01,NaN,data/interim/cbi_tehran_housing_monthly_1395_1...,missing_return,missing_previous_month_level
4,1395/02,1395,2,gold_18k,1395/01,"1,048,700.0000","1,030,700.0000",-0.0172,price_appreciation,1395-02-29,1395-01-31,data/raw/gold_18k/tgju_gold_18k_daily.csv,ok,NaN


## Housing data-integrity audit

Housing is repaired upstream, not in this notebook. The processed panel preserves the original extracted value, corrected level, official report, verification evidence, provenance, and quality flag. Returns are regenerated from adjacent levels.

The most severe defect occurred in `9707.pdf`: its damaged right-to-left text layer transformed the rendered Table 2 values `80958` and `86109` thousand IRR/mÂ² into malformed strings. The extraction workbook consequently stored Mehr 1397 as `0.6811` million IRR/mÂ². Visual review of `9707.pdf` and the repeated value in `9708.pdf` establish the valid level as `86.109`.

The following tables disclose the upstream adjudications and quantify their effect. They do not modify data.

In [4]:
housing_levels = levels_raw.loc[levels_raw["asset_id"].eq("tehran_housing")].sort_values(["jalali_year", "jalali_month"]).copy()

adjudicated_levels = housing_levels.loc[
    housing_levels["source_method"].eq("CBI_source_adjudicated_override")
    & ~np.isclose(housing_levels["month_end_level"], housing_levels["raw_source_level"], equal_nan=True),
    ["jalali_period", "raw_source_level", "month_end_level", "source_report", "data_quality_flag", "audit_note"],
]
revision_flags = housing_audit.loc[
    housing_audit["quality_flag"].str.contains("revision_disagreement", na=False),
    ["jalali_period", "price_level", "quality_flag", "verification_source"],
]
display(adjudicated_levels)
display(revision_flags)

,jalali_period,raw_source_level,month_end_level,source_report,data_quality_flag,audit_note
99,1396/12,48.9870,57.5910,data/raw/housing/cbi/reports/96012.pdf,corrected_official_current_report_adjacent_rev...,original=48.987; corrected=57.591; verificatio...
103,1397/01,55.2200,55.2800,data/raw/housing/cbi/reports/97.1.pdf,corrected_adjacent_official_reports_agree,original=55.22; corrected=55.280; verification...
115,1397/04,91.7280,69.7280,data/raw/housing/cbi/reports/9704.pdf,corrected_adjacent_official_reports_agree,original=91.728; corrected=69.728; verificatio...
127,1397/07,0.6811,86.1090,data/raw/housing/cbi/reports/9707.pdf,corrected_adjacent_official_reports_agree,original=0.6811; corrected=86.109; verificatio...


,jalali_period,price_level,quality_flag,verification_source
23,1396/12,57.5910,corrected_official_current_report_adjacent_rev...,data/raw/housing/cbi/reports/97.1.pdf
43,1398/08,126.8320,official_adjacent_revision_disagreement_curren...,data/raw/housing/cbi/reports/9809.pdf


In [5]:
def housing_year_statistics(level_column, mehr_override=None):
    series = housing_levels.set_index("jalali_period")[level_column].copy()
    if mehr_override is not None:
        series.loc["1397/07"] = mehr_override
    periods = [f"1397/{month:02d}" for month in range(1, 13)]
    values = []
    for period in periods:
        month = int(period[-2:])
        previous = "1396/12" if month == 1 else f"1397/{month - 1:02d}"
        values.append(series.loc[period] / series.loc[previous] - 1)
    values = np.asarray(values)
    return {"compounded_return": np.prod(1 + values) - 1, "monthly_standard_deviation": values.std(ddof=1), "annualized_volatility": values.std(ddof=1) * np.sqrt(12)}

housing_1397_comparison = pd.DataFrame({
    "Original extraction": housing_year_statistics("raw_source_level"),
    "Only Mehr corrected": housing_year_statistics("raw_source_level", 86.109),
    "All verified corrections": housing_year_statistics("month_end_level"),
}).T
housing_1397_sequence = housing_audit.loc[
    housing_audit["jalali_period"].isin(["1397/05", "1397/06", "1397/07", "1397/08"]),
    ["jalali_period", "price_level", "monthly_return", "quality_flag"],
]
display(housing_1397_sequence)
display((housing_1397_comparison * 100).round(2).rename(columns=lambda name: f"{name} (%)"))

,jalali_period,price_level,monthly_return,quality_flag
28,1397/05,73.9990,0.0613,ok
29,1397/06,80.9580,0.0940,ok
30,1397/07,86.1090,0.0636,corrected_adjacent_official_reports_agree
31,1397/08,91.7940,0.0660,ok


,compounded_return (%),monthly_standard_deviation (%),annualized_volatility (%)
Original extraction,125.3900,"3,862.3500","13,379.5600"
Only Mehr corrected,125.3900,13.3100,46.1200
All verified corrections,91.7200,4.0600,14.0800


Correcting Mehr alone reduces 1397 annualized housing volatility from roughly **13,380%** to **46.1%**, showing that the malformed level generated almost all of the original instability. Applying every source-verified correction reduces annualized volatility to **14.1%**. The corrected Mordad-Aban returns are ordinary single-digit changes; no smoothing, interpolation, or zero filling is used.

## Canonical monthly analytical panel

All assets, including housing, come directly from the canonical processed return panel. Missing returns remain missing and no interpolation is introduced to force eligibility. The notebook validates and analyzes data; it contains no override.

In [6]:
monthly = (
    returns_raw.pivot(index=["jalali_year", "jalali_month", "jalali_period"], columns="asset_id", values="monthly_return")
    .reset_index().sort_values(["jalali_year", "jalali_month"]).reset_index(drop=True)
)
monthly.head()

asset_id,jalali_year,jalali_month,jalali_period,equity_tedpix,fixed_income_etemad,gold_18k,tehran_housing
0,1395,1,1395/01,-0.0223,0.0273,0.0133,NaN
1,1395,2,1395/02,-0.0257,0.0183,-0.0172,0.0164
2,1395,3,1395/03,-0.0497,0.0167,0.0150,0.0095
3,1395,4,1395/04,0.0262,0.0185,0.0589,0.0051
4,1395,5,1395/05,0.0479,0.0176,0.0214,0.0095


In [7]:
analysis_assets = RISKY_ASSETS + [FIXED_INCOME]

coverage = (
    monthly
    .groupby("jalali_year")[analysis_assets]
    .count()
)

display(coverage)

for year in ANALYSIS_YEARS:
    expected = EXPECTED_MONTHS[year]
    assert (coverage.loc[year, analysis_assets] == expected).all(), (
        f"Incomplete analysis-period coverage detected in {year}; expected {expected} months."
    )


asset_id,gold_18k,equity_tedpix,tehran_housing,fixed_income_etemad
jalali_year,,,,
1395,12,12,11,12
1396,12,12,12,12
1397,12,12,12,12
1398,12,12,12,12
1399,12,12,12,12
1400,12,12,12,12
1401,12,12,12,12
1402,12,12,12,12
1403,12,12,12,12


## Realized return and risk by asset

Before solving the portfolio problem, each asset is examined on its own. Realized annual return is obtained by compounding monthly returns,

$$
R_{i,y}
=
\prod_{m=1}^{12}(1+r_{i,y,m})-1,
$$

while realized volatility is reported as the sample standard deviation of monthly returns annualized by $\sqrt{12}$,

$$
\sigma_{i,y}^{ann}
=
\sqrt{12}\,
s(r_{i,y,1},\ldots,r_{i,y,12}).
$$

This table is important for interpreting the optimizer. It makes clear whether a large portfolio weight is associated with unusually high realized return, unusually low volatility, or both.


In [8]:
def calculate_asset_year_statistics(
    monthly_df: pd.DataFrame,
    years: list[int],
    assets: list[str],
) -> pd.DataFrame:
    """Calculate period compounded return and annualized monthly volatility by asset."""
    rows = []

    for year in years:
        year_df = monthly_df.loc[monthly_df["jalali_year"] == year]

        for asset in assets:
            returns = year_df[asset].dropna().to_numpy(dtype=float)

            if len(returns) != 12:
                continue

            rows.append(
                {
                    "year": year,
                    "asset": asset,
                    "asset_name": ASSET_LABELS[asset],
                    "annual_return": np.prod(1.0 + returns) - 1.0,
                    "annualized_volatility": (
                        returns.std(ddof=1) * np.sqrt(12)
                    ),
                }
            )

    return pd.DataFrame(rows)


asset_year_stats = calculate_asset_year_statistics(
    monthly,
    ANALYSIS_YEARS,
    analysis_assets,
)

asset_year_table = (
    asset_year_stats
    .assign(
        annual_return=lambda df: df["annual_return"] * 100,
        annualized_volatility=lambda df: df["annualized_volatility"] * 100,
    )
    .pivot(
        index="year",
        columns="asset_name",
        values=["annual_return", "annualized_volatility"],
    )
)

asset_year_table.columns = [
    f"{asset} â€” {'Return (%)' if metric == 'annual_return' else 'Volatility (%)'}"
    for metric, asset in asset_year_table.columns
]

display(asset_year_table.round(2))


,Equity â€” Return (%),Fixed Income â€” Return (%),Gold â€” Return (%),Housing â€” Return (%),Equity â€” Volatility (%),Fixed Income â€” Volatility (%),Gold â€” Volatility (%),Housing â€” Volatility (%)
year,,,,,,,,
1396,24.6800,23.9100,32.9200,26.0900,9.8700,1.4100,13.0500,8.6800
1397,85.5400,22.2600,180.3400,91.7200,39.8600,0.8900,57.0000,14.0800
1398,187.0800,22.8500,40.1800,41.5500,20.5900,1.6000,18.8500,15.7500
1399,154.9600,21.5600,80.1400,93.7100,81.0800,7.3800,43.6400,17.1400
1400,4.5500,20.9800,13.6000,16.0000,26.3500,0.5200,19.0200,8.4000
1401,43.3900,23.4200,114.1400,85.7700,35.8900,0.3000,28.8600,15.7900
1402,11.9700,26.1700,23.2100,24.8300,27.5500,0.3200,22.3000,14.7300
1403,23.4600,31.5600,149.2400,15.0700,26.0300,0.5300,34.3300,5.9900
1404,37.0400,34.9500,117.5200,37.8400,43.1500,0.7600,45.5400,9.0200


In [9]:
plot_stats = asset_year_stats.assign(
    annual_return_pct=lambda df: df["annual_return"] * 100,
    annualized_volatility_pct=lambda df: df["annualized_volatility"] * 100,
)

fig = px.scatter(
    plot_stats,
    x="annualized_volatility_pct",
    y="annual_return_pct",
    color="asset_name",
    text="year",
    hover_data={
        "year": True,
        "annual_return_pct": ":.2f",
        "annualized_volatility_pct": ":.2f",
        "asset_name": False,
    },
    labels={
        "annualized_volatility_pct": "Annualized volatility (%)",
        "annual_return_pct": "Annual return (%)",
        "asset_name": "Asset",
    },
    title="Realized Return and Risk by Asset and Year",
)

fig.update_traces(textposition="top center")
fig.update_layout(legend_title_text="Asset")
fig.show()


## Stage I implementation

Stage I applies the framework defined at the beginning of the notebook. For every candidate initial weight vector, the code reconstructs the buy-and-hold wealth path of gold, equity, and housing, computes the realized monthly return of that risky sleeve, and subtracts the contemporaneous fixed-income benchmark return.

The optimization is long-only and fully invested within the risky sleeve. Multiple starting points are used so that the solution is not dependent on a single SLSQP initialization. The resulting weights are ex-post: they summarize the most efficient risky composition under return paths that are already known.

In [10]:
def get_year_data(
    monthly_df: pd.DataFrame,
    year: int,
    risky_assets: list[str],
    benchmark_asset: str,
) -> pd.DataFrame:
    """Return complete aligned observations for one full year or the 1405 YTD period."""
    columns = ["jalali_month"] + risky_assets + [benchmark_asset]

    year_df = (
        monthly_df.loc[monthly_df["jalali_year"] == year, columns]
        .dropna()
        .sort_values("jalali_month")
        .copy()
    )

    expected = EXPECTED_MONTHS[year]
    if len(year_df) != expected:
        raise ValueError(
            f"Year {year} does not contain {expected} aligned monthly observations."
        )

    return year_df


def simulate_buy_and_hold(
    initial_weights: np.ndarray,
    risky_returns: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Simulate monthly returns and ending weights for a buy-and-hold risky sleeve."""
    asset_wealth = np.asarray(initial_weights, dtype=float).copy()
    previous_portfolio_wealth = asset_wealth.sum()
    portfolio_returns = []

    for month_returns in risky_returns:
        asset_wealth *= 1.0 + month_returns
        portfolio_wealth = asset_wealth.sum()

        portfolio_returns.append(
            portfolio_wealth / previous_portfolio_wealth - 1.0
        )

        previous_portfolio_wealth = portfolio_wealth

    ending_weights = asset_wealth / asset_wealth.sum()

    return np.asarray(portfolio_returns), ending_weights


In [11]:
def benchmark_relative_score(
    weights: np.ndarray,
    risky_returns: np.ndarray,
    benchmark_returns: np.ndarray,
) -> float:
    """Return mean differential return divided by differential-return volatility."""
    portfolio_returns, _ = simulate_buy_and_hold(
        weights,
        risky_returns,
    )

    differential_returns = portfolio_returns - benchmark_returns
    tracking_error = differential_returns.std(ddof=1)

    if tracking_error <= 1e-12:
        return -np.inf

    return np.sqrt(12) * differential_returns.mean() / tracking_error


def negative_benchmark_relative_score(
    weights: np.ndarray,
    risky_returns: np.ndarray,
    benchmark_returns: np.ndarray,
) -> float:
    """Objective passed to scipy.optimize.minimize."""
    score = benchmark_relative_score(
        weights,
        risky_returns,
        benchmark_returns,
    )

    if not np.isfinite(score):
        return 1e12

    return -score


In [12]:
def optimize_risky_sleeve(
    year_df: pd.DataFrame,
    risky_assets: list[str],
    benchmark_asset: str,
    random_state: int = 42,
    n_random_starts: int = 30,
):
    """Estimate the long-only benchmark-relative risky sleeve using multi-start SLSQP."""
    risky_returns = year_df[risky_assets].to_numpy(dtype=float)
    benchmark_returns = year_df[benchmark_asset].to_numpy(dtype=float)

    n_assets = len(risky_assets)
    rng = np.random.default_rng(random_state)

    starting_points = [
        np.full(n_assets, 1.0 / n_assets),
        *np.eye(n_assets),
        *rng.dirichlet(np.ones(n_assets), size=n_random_starts),
    ]

    bounds = [(0.0, 1.0)] * n_assets
    constraint = {
        "type": "eq",
        "fun": lambda weights: weights.sum() - 1.0,
    }

    best_result = None

    for initial_weights in starting_points:
        result = minimize(
            negative_benchmark_relative_score,
            x0=initial_weights,
            args=(risky_returns, benchmark_returns),
            method="SLSQP",
            bounds=bounds,
            constraints=constraint,
            options={"maxiter": 2000, "ftol": 1e-12},
        )

        if not result.success:
            continue

        if (
            best_result is None
            or result.fun < best_result.fun
        ):
            best_result = result

    if best_result is None:
        raise RuntimeError("No successful optimization run was obtained.")

    if not np.isclose(best_result.x.sum(), 1.0, atol=1e-8):
        raise RuntimeError("Optimized weights do not sum to one.")

    if (best_result.x < -1e-10).any():
        raise RuntimeError("A negative risky-asset weight was produced.")

    return best_result


In [13]:
def stage_one_diagnostics(
    year_df: pd.DataFrame,
    risky_assets: list[str],
    benchmark_asset: str,
    initial_weights: np.ndarray,
) -> dict:
    """Calculate return, risk, score, and weight-drift diagnostics for Stage I."""
    risky_returns = year_df[risky_assets].to_numpy(dtype=float)
    benchmark_returns = year_df[benchmark_asset].to_numpy(dtype=float)

    portfolio_returns, ending_weights = simulate_buy_and_hold(
        initial_weights,
        risky_returns,
    )

    differential_returns = portfolio_returns - benchmark_returns

    return {
        "risky_sleeve_return": np.prod(1.0 + portfolio_returns) - 1.0,
        "benchmark_return": np.prod(1.0 + benchmark_returns) - 1.0,
        "annualized_volatility": (
            portfolio_returns.std(ddof=1) * np.sqrt(12)
        ),
        "annualized_tracking_error": (
            differential_returns.std(ddof=1) * np.sqrt(12)
        ),
        "benchmark_relative_score": (
            np.sqrt(12) * differential_returns.mean()
            / differential_returns.std(ddof=1)
        ),
        "ending_weights": ending_weights,
    }


def run_stage_one(
    monthly_df: pd.DataFrame,
    years: list[int],
    risky_assets: list[str],
    benchmark_asset: str,
) -> pd.DataFrame:
    """Solve Stage I separately for each Solar Hijri year."""
    rows = []

    for year in years:
        year_df = get_year_data(
            monthly_df,
            year,
            risky_assets,
            benchmark_asset,
        )

        result = optimize_risky_sleeve(
            year_df,
            risky_assets,
            benchmark_asset,
            random_state=year,
        )

        diagnostics = stage_one_diagnostics(
            year_df,
            risky_assets,
            benchmark_asset,
            result.x,
        )

        row = {
            "year": year,
            "benchmark_relative_score": diagnostics[
                "benchmark_relative_score"
            ],
            "risky_sleeve_return": diagnostics["risky_sleeve_return"],
            "benchmark_return": diagnostics["benchmark_return"],
            "annualized_volatility": diagnostics["annualized_volatility"],
            "annualized_tracking_error": diagnostics[
                "annualized_tracking_error"
            ],
        }

        for asset, initial_weight, ending_weight in zip(
            risky_assets,
            result.x,
            diagnostics["ending_weights"],
        ):
            row[f"initial_weight_{asset}"] = initial_weight
            row[f"ending_weight_{asset}"] = ending_weight

        rows.append(row)

    return pd.DataFrame(rows)


stage_one_results = run_stage_one(
    monthly,
    ANALYSIS_YEARS,
    RISKY_ASSETS,
    FIXED_INCOME,
)

def numerical_qa(year_df, weights, n_samples=25_000, seed=0):
    """Compare the selected solution with boundaries, equal weights, and random portfolios."""
    risky = year_df[RISKY_ASSETS].to_numpy(float)
    benchmark = year_df[FIXED_INCOME].to_numpy(float)
    rng = np.random.default_rng(seed)
    candidates = np.vstack([np.eye(len(RISKY_ASSETS)), np.full((1, len(RISKY_ASSETS)), 1 / len(RISKY_ASSETS)), rng.dirichlet(np.ones(len(RISKY_ASSETS)), n_samples)])
    growth = np.cumprod(1 + risky, axis=0)
    wealth = candidates @ growth.T
    prior = np.column_stack([np.ones(len(candidates)), wealth[:, :-1]])
    differential = wealth / prior - 1 - benchmark
    scores = np.sqrt(12) * differential.mean(axis=1) / differential.std(axis=1, ddof=1)
    selected = benchmark_relative_score(weights, risky, benchmark)
    return {"best_sampled_score": float(np.nanmax(scores)), "selected_minus_best_sampled": selected - float(np.nanmax(scores))}

qa_rows = []
for row in stage_one_results.itertuples(index=False):
    year_df = get_year_data(monthly, row.year, RISKY_ASSETS, FIXED_INCOME)
    weights = np.array([getattr(row, f"initial_weight_{asset}") for asset in RISKY_ASSETS])
    qa_rows.append({"year": row.year, **numerical_qa(year_df, weights, seed=row.year)})
stage_one_results = stage_one_results.merge(pd.DataFrame(qa_rows), on="year", how="left")
assert (stage_one_results["selected_minus_best_sampled"] >= -1e-8).all()

stage_one_results


,year,benchmark_relative_score,risky_sleeve_return,benchmark_return,annualized_volatility,annualized_tracking_error,initial_weight_gold_18k,ending_weight_gold_18k,initial_weight_equity_tedpix,ending_weight_equity_tedpix,initial_weight_tehran_housing,ending_weight_tehran_housing,best_sampled_score,selected_minus_best_sampled
0,1396,0.5857,0.3185,0.2391,0.1123,0.1175,0.8437,0.8505,0.0000,0.0000,0.1563,0.1495,0.5857,0.0000
1,1397,3.5792,0.9562,0.2226,0.1407,0.1386,0.0440,0.0631,0.0000,0.0000,0.9560,0.9369,3.5765,0.0027
2,1398,4.2431,1.8708,0.2285,0.2059,0.2151,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,4.2431,0.0000
3,1399,2.7921,0.9454,0.2156,0.1739,0.1784,0.0000,0.0000,0.0135,0.0177,0.9865,0.9823,2.7905,0.0016
4,1400,-0.2533,0.1360,0.2098,0.1902,0.1880,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,-0.2533,0.0000
5,1401,3.0203,0.9250,0.2342,0.1575,0.1561,0.2371,0.2638,0.0000,0.0000,0.7629,0.7362,3.0194,0.0010
6,1402,-0.0101,0.2321,0.2617,0.2230,0.2235,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,-0.0101,0.0000
7,1403,2.1119,1.4924,0.3156,0.3433,0.3413,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,2.1119,0.0000
8,1404,1.3071,1.1752,0.3495,0.4554,0.4537,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.3071,0.0000
9,1405,4.7412,0.5852,0.1489,0.1733,0.1753,0.1556,0.1171,0.1087,0.1119,0.7357,0.7710,4.7410,0.0002


## Stage I results

The table reports initial and ending risky-sleeve weights, realized returns, total volatility, tracking error, and the annualized benchmark-relative efficiency ratio. Ending weights differ because the portfolio is held without monthly rebalancing.

Each selected solution is checked against the three corner portfolios, equal weights, and 25,000 deterministic random simplex portfolios. `selected_minus_best_sampled` should be nonnegative within numerical tolerance.

In [14]:
stage_one_display = stage_one_results.copy()

percentage_columns = [
    column
    for column in stage_one_display.columns
    if column.startswith("initial_weight_")
    or column.startswith("ending_weight_")
    or column in {
        "risky_sleeve_return",
        "benchmark_return",
        "annualized_volatility",
        "annualized_tracking_error",
    }
]

stage_one_display[percentage_columns] = (
    stage_one_display[percentage_columns] * 100
)

display(stage_one_display.round(2))


,year,benchmark_relative_score,risky_sleeve_return,benchmark_return,annualized_volatility,annualized_tracking_error,initial_weight_gold_18k,ending_weight_gold_18k,initial_weight_equity_tedpix,ending_weight_equity_tedpix,initial_weight_tehran_housing,ending_weight_tehran_housing,best_sampled_score,selected_minus_best_sampled
0,1396,0.5900,31.8500,23.9100,11.2300,11.7500,84.3700,85.0500,0.0000,0.0000,15.6300,14.9500,0.5900,0.0000
1,1397,3.5800,95.6200,22.2600,14.0700,13.8600,4.4000,6.3100,0.0000,0.0000,95.6000,93.6900,3.5800,0.0000
2,1398,4.2400,187.0800,22.8500,20.5900,21.5100,0.0000,0.0000,100.0000,100.0000,0.0000,0.0000,4.2400,0.0000
3,1399,2.7900,94.5400,21.5600,17.3900,17.8400,0.0000,0.0000,1.3500,1.7700,98.6500,98.2300,2.7900,0.0000
4,1400,-0.2500,13.6000,20.9800,19.0200,18.8000,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,-0.2500,0.0000
5,1401,3.0200,92.5000,23.4200,15.7500,15.6100,23.7100,26.3800,0.0000,0.0000,76.2900,73.6200,3.0200,0.0000
6,1402,-0.0100,23.2100,26.1700,22.3000,22.3500,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,-0.0100,0.0000
7,1403,2.1100,149.2400,31.5600,34.3300,34.1300,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,2.1100,0.0000
8,1404,1.3100,117.5200,34.9500,45.5400,45.3700,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,1.3100,0.0000
9,1405,4.7400,58.5200,14.8900,17.3300,17.5300,15.5600,11.7100,10.8700,11.1900,73.5700,77.1000,4.7400,0.0000


In [15]:
weight_columns = [
    f"initial_weight_{asset}"
    for asset in RISKY_ASSETS
]

weights_long = (
    stage_one_results[
        ["year"] + weight_columns
    ]
    .melt(
        id_vars="year",
        var_name="asset",
        value_name="weight",
    )
)

weights_long["asset"] = (
    weights_long["asset"]
    .str.replace("initial_weight_", "", regex=False)
    .map(ASSET_LABELS)
)

weights_long["weight_pct"] = weights_long["weight"] * 100

fig = px.bar(
    weights_long,
    x="year",
    y="weight_pct",
    color="asset",
    labels={
        "year": "Solar Hijri Year",
        "weight_pct": "Initial risky-sleeve weight (%)",
        "asset": "Asset",
    },
    title="Ex-Post Stage I Risky-Sleeve Allocation",
)

fig.update_layout(
    barmode="stack",
    legend_title_text="Asset",
    yaxis_range=[0, 100],
)

fig.show()


In [16]:
stage_one_summary = stage_one_results.copy()

initial_weight_columns = [
    f"initial_weight_{asset}"
    for asset in RISKY_ASSETS
]

stage_one_summary["dominant_asset"] = (
    stage_one_summary[initial_weight_columns]
    .idxmax(axis=1)
    .str.replace("initial_weight_", "", regex=False)
    .map(ASSET_LABELS)
)

stage_one_summary["dominant_weight_pct"] = (
    stage_one_summary[initial_weight_columns].max(axis=1) * 100
)

stage_one_summary["risky_return_pct"] = stage_one_summary["risky_sleeve_return"] * 100
stage_one_summary["annualized_tracking_error_pct"] = (
    stage_one_summary["annualized_tracking_error"] * 100
)

display(
    stage_one_summary[
        [
            "year",
            "dominant_asset",
            "dominant_weight_pct",
            "risky_return_pct",
            "annualized_tracking_error_pct",
            "benchmark_relative_score",
        ]
    ].round(2)
)


,year,dominant_asset,dominant_weight_pct,risky_return_pct,annualized_tracking_error_pct,benchmark_relative_score
0,1396,Gold,84.3700,31.8500,11.7500,0.5900
1,1397,Housing,95.6000,95.6200,13.8600,3.5800
2,1398,Equity,100.0000,187.0800,21.5100,4.2400
3,1399,Housing,98.6500,94.5400,17.8400,2.7900
4,1400,Gold,100.0000,13.6000,18.8000,-0.2500
5,1401,Housing,76.2900,92.5000,15.6100,3.0200
6,1402,Gold,100.0000,23.2100,22.3500,-0.0100
7,1403,Gold,100.0000,149.2400,34.1300,2.1100
8,1404,Gold,100.0000,117.5200,45.3700,1.3100
9,1405,Housing,73.5700,58.5200,17.5300,4.7400


## Interpretation of Stage I

Optimized weights must be read with the asset diagnostics. A large weight can reflect strong realized return, low benchmark-relative variation, diversification, or a combination. Boundary solutions are not automatically solver failures; the numerical comparison checks them against corners, equal weights, and a broad random sample.

These are hindsight allocations. They support historical regime comparison and model diagnostics, but they are not weights an investor could have known at the beginning of the year.

The solutions are strongly regime-dependent. Gold dominates 1396, 1400, 1402, 1403, and 1404; housing dominates 1397, 1399, 1401, and 1405 YTD; equity dominates only 1398. Five of ten solutions are exact corner portfolios, so the model does not identify a stable all-weather mix. The negative scores in 1400 and 1402 mean that even the best risky sleeve underperformed fixed income on the benchmark-relative objective in those realized samples.

## Stage II - total-portfolio allocation

Stage II reports sensitivity to an explicit risk-aversion parameter. It combines the Stage I sleeve with fixed income for each value in a documented grid. The results remain ex-post diagnostics; no single row is presented as an investor-specific recommendation.

### Risk-Aversion Sensitivity Analysis

Stage I identified the most efficient risky-asset composition for each historical year. Stage II keeps that risky composition fixed and determines how much of total wealth should be allocated to it.

Let $\alpha$ denote the fraction of initial wealth invested in the Stage I risky portfolio. The remaining share,

$$
1-\alpha,
$$

is invested in fixed income.

If the Stage I risky weights for year $y$ are collected in $w_{T,y}^{*}$, the initial total-portfolio weights become

$$
w_{y}^{total}(\alpha)
=
\left(
\alpha w_{T,y}^{*},
1-\alpha
\right).
$$

The portfolio remains buy-and-hold within the year. The initial weights are therefore allowed to drift naturally as the assets realize different monthly returns.

For a given value of the risk-aversion coefficient $\gamma$, portfolio utility is defined as

$$
U_y(\alpha;\gamma)
=
R_y(\alpha)
-
\frac{\gamma}{2}
\sigma_y^2(\alpha),
$$

where $R_y(\alpha)$ is the realized compounded return of the complete portfolio and $\sigma_y(\alpha)$ is its annualized realized volatility.

Stage II therefore solves

$$
\alpha_y^*(\gamma)
=
\arg\max_{0\leq\alpha\leq1}
\left[
R_y(\alpha)
-
\frac{\gamma}{2}
\sigma_y^2(\alpha)
\right].
$$

The analysis is repeated over a range of risk preferences:

$$
\Gamma
=
\{0,1,2,4,6,8,10,15\}.
$$

The case $\gamma=0$ represents a risk-neutral investor. As $\gamma$ increases, volatility receives a progressively larger penalty in the utility function.

The same value of $\gamma$ is interpreted consistently across years. Thus changes in optimal risky exposure reflect changes in the historical investment opportunity set rather than changes in assumed investor preferences.

In [17]:
GAMMA_VALUES = [0, 1, 2, 4, 6, 8, 10, 15, 20, 25, 30, 35, 40, 45, 50]

### Total-Portfolio Wealth Path

Stage II preserves the annual buy-and-hold assumption used throughout the analysis.

At the beginning of the year, the risky assets receive a total allocation of $\alpha$, distributed according to the Stage I optimal risky weights. Fixed income receives the remaining allocation $1-\alpha$.

If the initial risky weights are $w_i^*$, the initial wealth invested in risky asset $i$ is

$$
V_{i,0}
=
\alpha w_i^*,
$$

while initial fixed-income wealth is

$$
V_{f,0}
=
1-\alpha.
$$

Each position then evolves according to its own monthly return:

$$
V_{i,m}
=
V_{i,m-1}(1+r_{i,m}).
$$

Total portfolio wealth is

$$
V_{p,m}
=
\sum_i V_{i,m}.
$$

The realized monthly portfolio return is therefore

$$
r_{p,m}
=
\frac{V_{p,m}}{V_{p,m-1}}-1.
$$

This construction avoids imposing artificial monthly rebalancing.

In [18]:
def simulate_stage_two_portfolio(
    year_df: pd.DataFrame,
    risky_assets: list[str],
    risky_weights: np.ndarray,
    alpha: float,
    fixed_income_asset: str = FIXED_INCOME,
) -> dict:
    """
    Simulate a buy-and-hold portfolio consisting of the Stage I risky sleeve
    and fixed income.

    Parameters
    ----------
    year_df
        Monthly returns for one Solar Hijri year.
    risky_assets
        Risky assets included in the Stage I portfolio.
    risky_weights
        Stage I risky-asset weights, summing to one.
    alpha
        Fraction of total initial wealth allocated to the risky sleeve.
    fixed_income_asset
        Fixed-income asset used for the remaining allocation.

    Returns
    -------
    dict
        Portfolio return, realized volatility, ending wealth,
        monthly returns, and final asset weights.
    """
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must lie between 0 and 1.")

    risky_weights = np.asarray(risky_weights, dtype=float)

    if not np.isclose(risky_weights.sum(), 1.0):
        raise ValueError("Stage I risky weights must sum to one.")

    initial_risky_wealth = alpha * risky_weights
    initial_fixed_income_wealth = 1.0 - alpha

    asset_wealth = np.concatenate(
        [initial_risky_wealth, [initial_fixed_income_wealth]]
    )

    asset_columns = risky_assets + [fixed_income_asset]
    monthly_returns = year_df[asset_columns].to_numpy(dtype=float)

    previous_portfolio_wealth = asset_wealth.sum()
    portfolio_returns = []

    for month_returns in monthly_returns:
        asset_wealth *= 1.0 + month_returns

        portfolio_wealth = asset_wealth.sum()

        portfolio_return = (
            portfolio_wealth / previous_portfolio_wealth - 1.0
        )

        portfolio_returns.append(portfolio_return)
        previous_portfolio_wealth = portfolio_wealth

    portfolio_returns = np.asarray(portfolio_returns)

    annual_return = asset_wealth.sum() - 1.0
    monthly_volatility = portfolio_returns.std(ddof=1)
    annualized_volatility = monthly_volatility * np.sqrt(12)

    final_weights = asset_wealth / asset_wealth.sum()

    return {
        "annual_return": annual_return,
        "monthly_volatility": monthly_volatility,
        "annualized_volatility": annualized_volatility,
        "ending_wealth": asset_wealth.sum(),
        "monthly_returns": portfolio_returns,
        "final_weights": final_weights,
    }

### Utility

For every candidate value of $\alpha$, realized return and realized volatility are obtained directly from the buy-and-hold wealth path.

Utility is then evaluated as

$$
U_y(\alpha;\gamma)
=
R_y(\alpha)
-
\frac{\gamma}{2}
\sigma_y^2(\alpha).
$$

Returns and volatility enter the calculation in decimal form. For example, a return of 25% is represented as $0.25$, not $25$.

When $\gamma=0$, the volatility penalty disappears and the problem reduces to maximizing realized return alone.

In [19]:
def stage_two_utility(
    year_df: pd.DataFrame,
    risky_assets: list[str],
    risky_weights: np.ndarray,
    alpha: float,
    gamma: float,
    fixed_income_asset: str = FIXED_INCOME,
) -> float:
    """
    Evaluate realized mean-variance utility for one risky allocation.
    """
    if gamma < 0:
        raise ValueError("gamma cannot be negative.")

    portfolio = simulate_stage_two_portfolio(
        year_df=year_df,
        risky_assets=risky_assets,
        risky_weights=risky_weights,
        alpha=alpha,
        fixed_income_asset=fixed_income_asset,
    )

    annual_return = portfolio["annual_return"]
    annualized_variance = portfolio["annualized_volatility"] ** 2

    return annual_return - 0.5 * gamma * annualized_variance

### Finding the Optimal Risky Share

Because Stage II contains only one decision variable, $\alpha$, the optimization is numerically simple.

A dense grid is used over the interval

$$
0\leq\alpha\leq1.
$$

This approach is transparent and avoids relying on local numerical behavior. For each value of $\gamma$, every candidate risky allocation is evaluated using the complete buy-and-hold wealth path, and the allocation with the highest utility is retained.

In [20]:
def optimize_stage_two_alpha(
    year_df: pd.DataFrame,
    risky_assets: list[str],
    risky_weights: np.ndarray,
    gamma: float,
    fixed_income_asset: str = FIXED_INCOME,
    grid_size: int = 5001,
) -> dict:
    """
    Find the utility-maximizing risky allocation for a given gamma
    using a dense deterministic grid search.
    """
    alpha_grid = np.linspace(0.0, 1.0, grid_size)

    risky_growth = np.cumprod(
        1 + year_df[risky_assets].to_numpy(float), axis=0
    )
    risky_sleeve_wealth = risky_growth @ np.asarray(risky_weights, dtype=float)
    fixed_income_wealth = np.cumprod(
        1 + year_df[fixed_income_asset].to_numpy(float)
    )
    wealth_grid = (
        alpha_grid[:, None] * risky_sleeve_wealth[None, :]
        + (1 - alpha_grid[:, None]) * fixed_income_wealth[None, :]
    )
    prior_wealth_grid = np.column_stack(
        [np.ones(grid_size), wealth_grid[:, :-1]]
    )
    return_grid = wealth_grid / prior_wealth_grid - 1
    annual_return_grid = wealth_grid[:, -1] - 1
    annualized_volatility_grid = return_grid.std(axis=1, ddof=1) * np.sqrt(12)
    utilities = (
        annual_return_grid
        - 0.5 * gamma * annualized_volatility_grid**2
    )

    best_index = np.argmax(utilities)
    best_alpha = alpha_grid[best_index]

    portfolio = simulate_stage_two_portfolio(
        year_df=year_df,
        risky_assets=risky_assets,
        risky_weights=risky_weights,
        alpha=best_alpha,
        fixed_income_asset=fixed_income_asset,
    )

    return {
        "gamma": gamma,
        "alpha": best_alpha,
        "fixed_income_weight": 1.0 - best_alpha,
        "annual_return": portfolio["annual_return"],
        "annualized_volatility": portfolio["annualized_volatility"],
        "utility": utilities[best_index],
        "ending_wealth": portfolio["ending_wealth"],
        "final_weights": portfolio["final_weights"],
    }

### Sensitivity Across Risk Preferences

The optimization is now repeated for every historical year and every value of $\gamma$.

The Stage I risky composition remains unchanged within each year. Stage II changes only the total exposure to that risky sleeve.

This distinction allows the results to answer two separate questions cleanly: Stage I determines *what to own inside the risky portfolio*, while Stage II determines *how much of that portfolio to own*.

In [21]:
def run_stage_two_sensitivity(
    monthly_df: pd.DataFrame,
    stage_one_results: pd.DataFrame,
    risky_assets: list[str],
    gamma_values: list[float],
    fixed_income_asset: str = FIXED_INCOME,
) -> pd.DataFrame:
    """
    Run Stage II risk-aversion sensitivity analysis for every year.
    """
    rows = []

    weight_columns = [
        f"initial_weight_{asset}"
        for asset in risky_assets
    ]

    for _, stage_one_row in stage_one_results.iterrows():
        year = int(stage_one_row["year"])

        year_df = get_year_data(
            monthly_df,
            year,
            risky_assets,
            fixed_income_asset,
        )

        risky_weights = (
            stage_one_row[weight_columns]
            .to_numpy(dtype=float)
        )

        for gamma in gamma_values:
            result = optimize_stage_two_alpha(
                year_df=year_df,
                risky_assets=risky_assets,
                risky_weights=risky_weights,
                gamma=gamma,
                fixed_income_asset=fixed_income_asset,
            )

            row = {
                "year": year,
                "gamma": gamma,
                "alpha": result["alpha"],
                "fixed_income_weight": result["fixed_income_weight"],
                "annual_return": result["annual_return"],
                "annualized_volatility": result["annualized_volatility"],
                "utility": result["utility"],
                "ending_wealth": result["ending_wealth"],
            }

            for asset, stage_one_weight in zip(
                risky_assets,
                risky_weights,
            ):
                row[f"stage_one_weight_{asset}"] = stage_one_weight
                row[f"total_weight_{asset}"] = (
                    result["alpha"] * stage_one_weight
                )

            rows.append(row)

    return pd.DataFrame(rows)

In [22]:
stage_two_results = run_stage_two_sensitivity(
    monthly_df=monthly,
    stage_one_results=stage_one_results,
    risky_assets=RISKY_ASSETS,
    gamma_values=GAMMA_VALUES,
)

stage_two_results.head(16)

,year,gamma,alpha,fixed_income_weight,annual_return,annualized_volatility,utility,ending_wealth,stage_one_weight_gold_18k,total_weight_gold_18k,stage_one_weight_equity_tedpix,total_weight_equity_tedpix,stage_one_weight_tehran_housing,total_weight_tehran_housing
0,1396,0,1.0000,0.0000,0.3185,0.1123,0.3185,1.3185,0.8437,0.8437,0.0000,0.0000,0.1563,0.1563
1,1396,1,1.0000,0.0000,0.3185,0.1123,0.3122,1.3185,0.8437,0.8437,0.0000,0.0000,0.1563,0.1563
2,1396,2,1.0000,0.0000,0.3185,0.1123,0.3059,1.3185,0.8437,0.8437,0.0000,0.0000,0.1563,0.1563
3,1396,4,1.0000,0.0000,0.3185,0.1123,0.2933,1.3185,0.8437,0.8437,0.0000,0.0000,0.1563,0.1563
4,1396,6,0.9732,0.0268,0.3164,0.1091,0.2807,1.3164,0.8437,0.8211,0.0000,0.0000,0.1563,0.1521
5,1396,8,0.7618,0.2382,0.2996,0.0837,0.2716,1.2996,0.8437,0.6427,0.0000,0.0000,0.1563,0.1191
6,1396,10,0.6294,0.3706,0.2891,0.0682,0.2659,1.2891,0.8437,0.5310,0.0000,0.0000,0.1563,0.0984
7,1396,15,0.4456,0.5544,0.2745,0.0471,0.2579,1.2745,0.8437,0.3759,0.0000,0.0000,0.1563,0.0697
8,1396,20,0.3504,0.6496,0.2669,0.0366,0.2536,1.2669,0.8437,0.2956,0.0000,0.0000,0.1563,0.0548
9,1396,25,0.2922,0.7078,0.2623,0.0304,0.2508,1.2623,0.8437,0.2465,0.0000,0.0000,0.1563,0.0457


### Reading the Sensitivity Results

The value of $\alpha$ measures the optimal total exposure to risky assets.

An estimate of

$$
\alpha=1
$$

means that the utility-maximizing portfolio is fully invested in the Stage I risky sleeve.

An estimate of

$$
\alpha=0
$$

means that the entire portfolio is allocated to fixed income.

Intermediate values create a combination of the Stage I risky sleeve and fixed income. As risk aversion increases, the volatility penalty becomes stronger and the optimal value of $\alpha$ will generally decline, although the precise response depends on the realized return and risk environment of each year.

In the realized sample, 1400 and 1402 select fixed income at every tested value of $\gamma$ because their Stage I sleeves returned less than the benchmark. By contrast, 1397, 1398, 1399, and 1401 remain fully risky even at $\gamma=15$ because their realized return advantage dominates the variance penalty. The most informative interior responses occur in 1396 and 1404: at $\gamma=15$, risky exposure falls to 44.56% and 19.56%, respectively. The apparent stability in several other years is therefore not universal robustness; it reflects unusually large ex-post nominal return spreads.

In [23]:
stage_two_display = stage_two_results[
    [
        "year",
        "gamma",
        "alpha",
        "fixed_income_weight",
        "annual_return",
        "annualized_volatility",
        "utility",
    ]
].copy()

stage_two_display["alpha"] *= 100
stage_two_display["fixed_income_weight"] *= 100
stage_two_display["annual_return"] *= 100
stage_two_display["annualized_volatility"] *= 100

stage_two_display.rename(
    columns={
        "alpha": "Risky Allocation (%)",
        "fixed_income_weight": "Fixed Income (%)",
        "annual_return": "Portfolio Return (%)",
        "annualized_volatility": "Annualized Volatility (%)",
        "utility": "Utility",
    }
).round(2)

,year,gamma,Risky Allocation (%),Fixed Income (%),Portfolio Return (%),Annualized Volatility (%),Utility
0,1396,0,100.0000,0.0000,31.8500,11.2300,0.3200
1,1396,1,100.0000,0.0000,31.8500,11.2300,0.3100
2,1396,2,100.0000,0.0000,31.8500,11.2300,0.3100
3,1396,4,100.0000,0.0000,31.8500,11.2300,0.2900
4,1396,6,97.3200,2.6800,31.6400,10.9100,0.2800
...,...,...,...,...,...,...,...
145,1405,30,47.9400,52.0600,35.8100,8.3200,0.2500
146,1405,35,41.0200,58.9800,32.7900,7.1100,0.2400
147,1405,40,35.8600,64.1400,30.5400,6.2000,0.2300
148,1405,45,31.8800,68.1200,28.8000,5.5000,0.2200


### Risk-Aversion Sensitivity

The following figure traces the optimal risky allocation as risk aversion increases.

Each line represents one historical year. Differences in the shape and level of these curves reveal how strongly the optimal total allocation depended on the realized investment environment of that year.

In [24]:
stage_two_plot = stage_two_results.copy()
stage_two_plot["alpha_pct"] = stage_two_plot["alpha"] * 100

fig = px.line(
    stage_two_plot,
    x="gamma",
    y="alpha_pct",
    color="year",
    markers=True,
    labels={
        "gamma": "Risk Aversion (γ)",
        "alpha_pct": "Optimal Risky Allocation (%)",
        "year": "Solar Hijri Year",
    },
    title="Optimal Risky Allocation Across Risk-Aversion Levels",
)

fig.update_layout(
    hovermode="x unified",
    yaxis_range=[0, 100],
)

fig.show()

In [25]:
alpha_heatmap = (
    stage_two_results
    .pivot(
        index="year",
        columns="gamma",
        values="alpha",
    )
    * 100
)

fig = px.imshow(
    alpha_heatmap,
    text_auto=".1f",
    aspect="auto",
    labels={
        "x": "Risk Aversion (γ)",
        "y": "Solar Hijri Year",
        "color": "Risky Allocation (%)",
    },
    title="Optimal Risky Allocation by Year and Risk Aversion",
)

fig.show()

## Final Portfolio Weights

Once $\alpha_y^*(\gamma)$ has been determined, the total portfolio weights follow directly.

For risky asset $i$,

$$
w_{i,y}^{total}
=
\alpha_y^*(\gamma)
w_{i,y}^{Stage\,I},
$$

while the fixed-income weight is

$$
w_{f,y}^{total}
=
1-\alpha_y^*(\gamma).
$$

The final portfolio therefore preserves the relative composition discovered in Stage I while adjusting the overall amount of market risk according to the investor's degree of risk aversion.

In [26]:
total_weight_columns = [
    f"total_weight_{asset}"
    for asset in RISKY_ASSETS
]

final_portfolios = stage_two_results[
    [
        "year",
        "gamma",
        *total_weight_columns,
        "fixed_income_weight",
        "annual_return",
        "annualized_volatility",
        "utility",
    ]
].copy()

final_portfolios

,year,gamma,total_weight_gold_18k,total_weight_equity_tedpix,total_weight_tehran_housing,fixed_income_weight,annual_return,annualized_volatility,utility
0,1396,0,0.8437,0.0000,0.1563,0.0000,0.3185,0.1123,0.3185
1,1396,1,0.8437,0.0000,0.1563,0.0000,0.3185,0.1123,0.3122
2,1396,2,0.8437,0.0000,0.1563,0.0000,0.3185,0.1123,0.3059
3,1396,4,0.8437,0.0000,0.1563,0.0000,0.3185,0.1123,0.2933
4,1396,6,0.8211,0.0000,0.1521,0.0268,0.3164,0.1091,0.2807
...,...,...,...,...,...,...,...,...,...
145,1405,30,0.0746,0.0521,0.3527,0.5206,0.3581,0.0832,0.2543
146,1405,35,0.0638,0.0446,0.3018,0.5898,0.3279,0.0711,0.2395
147,1405,40,0.0558,0.0390,0.2638,0.6414,0.3054,0.0620,0.2285
148,1405,45,0.0496,0.0346,0.2345,0.6812,0.2880,0.0550,0.2199


In [27]:
REPRESENTATIVE_GAMMA = 4

gamma_4_portfolios = (
    final_portfolios
    .loc[
        final_portfolios["gamma"] == REPRESENTATIVE_GAMMA
    ]
    .copy()
)

gamma_4_portfolios

,year,gamma,total_weight_gold_18k,total_weight_equity_tedpix,total_weight_tehran_housing,fixed_income_weight,annual_return,annualized_volatility,utility
3,1396,4,0.8437,0.0000,0.1563,0.0000,0.3185,0.1123,0.2933
18,1397,4,0.0440,0.0000,0.9560,0.0000,0.9562,0.1407,0.9166
33,1398,4,0.0000,1.0000,0.0000,0.0000,1.8708,0.2059,1.7860
48,1399,4,0.0000,0.0135,0.9865,0.0000,0.9454,0.1739,0.8849
63,1400,4,0.0000,0.0000,0.0000,1.0000,0.2098,0.0052,0.2098
78,1401,4,0.2371,0.0000,0.7629,0.0000,0.9250,0.1575,0.8754
93,1402,4,0.0000,0.0000,0.0000,1.0000,0.2617,0.0032,0.2617
108,1403,4,1.0000,0.0000,0.0000,0.0000,1.4924,0.3433,1.2567
123,1404,4,1.0000,0.0000,0.0000,0.0000,1.1752,0.4554,0.7604
138,1405,4,0.1556,0.1087,0.7357,0.0000,0.5852,0.1733,0.5252


In [28]:
gamma_4_display = gamma_4_portfolios.copy()

percentage_columns = (
    total_weight_columns
    + [
        "fixed_income_weight",
        "annual_return",
        "annualized_volatility",
    ]
)

gamma_4_display[percentage_columns] *= 100

gamma_4_display.round(2)

,year,gamma,total_weight_gold_18k,total_weight_equity_tedpix,total_weight_tehran_housing,fixed_income_weight,annual_return,annualized_volatility,utility
3,1396,4,84.3700,0.0000,15.6300,0.0000,31.8500,11.2300,0.2900
18,1397,4,4.4000,0.0000,95.6000,0.0000,95.6200,14.0700,0.9200
33,1398,4,0.0000,100.0000,0.0000,0.0000,187.0800,20.5900,1.7900
48,1399,4,0.0000,1.3500,98.6500,0.0000,94.5400,17.3900,0.8800
63,1400,4,0.0000,0.0000,0.0000,100.0000,20.9800,0.5200,0.2100
78,1401,4,23.7100,0.0000,76.2900,0.0000,92.5000,15.7500,0.8800
93,1402,4,0.0000,0.0000,0.0000,100.0000,26.1700,0.3200,0.2600
108,1403,4,100.0000,0.0000,0.0000,0.0000,149.2400,34.3300,1.2600
123,1404,4,100.0000,0.0000,0.0000,0.0000,117.5200,45.5400,0.7600
138,1405,4,15.5600,10.8700,73.5700,0.0000,58.5200,17.3300,0.5300


## Methodological limitations

- Housing is the CBI average transaction price per square metre, not a constant-quality repeat-sales index; transaction composition can affect returns.
- Housing returns exclude rent, maintenance, vacancy, taxes, transaction costs, and illiquidity.
- TEDPIX is an index rather than a directly tradable portfolio; implementation costs and tracking error are omitted.
- Etemad is an ETF rather than a theoretical risk-free asset. Its distribution and corporate-action treatment remains under source audit.
- Each full-year optimization uses only 12 observations; the 1405 YTD estimate uses five. Results are in-sample, path-sensitive, and unsuitable as stand-alone forecasts.
- Housing changes from CBI to chain-linked Kilid after 1403/05. This permits continued return and volatility estimation, but comparisons across the source boundary carry measurement-regime uncertainty.

## Reproducibility

From the project directory, regenerate and validate inputs before running the notebook:

```powershell
$env:PYTHONPATH = "src"
python -m asset_allocation.build_monthly_return_panel
python -m pytest -q
```

The notebook reads only regenerated processed panels and the housing-quality audit. It does not modify source or processed data.

# Alternative Analysis: Full-Sample Fixed Volatility

## Definition and purpose

This section repeats the two-stage analysis using the requested alternative definition of volatility. Instead of estimating risk separately inside each year, the risk model is estimated **once from every aligned monthly observation in 1396–1405/05** and then held fixed across years. Annual returns still vary by year.

For asset returns $r_m$, fixed annualized volatility is

$$\sigma^{fixed}=\sqrt{12}\,s(r_1,\ldots,r_T).$$

Portfolio risk must include co-movement, not only individual standard deviations. Therefore the analysis estimates one full-sample annualized covariance matrix $\Sigma^{fixed}=12\operatorname{Cov}(r_m)$ and uses

$$\sigma_p^{fixed}(w)=\sqrt{w^\top\Sigma^{fixed}w}. $$

For Stage I, the covariance matrix is calculated from each risky asset's monthly return minus the contemporaneous fixed-income return. This creates a fixed full-sample tracking-error model. This specification intentionally uses the complete sample and therefore contains look-ahead information; it is a comparative ex-post sensitivity analysis, not a forecasting model.

In [29]:
fixed_risk_sample = (
    monthly.loc[
        monthly["jalali_year"].isin(ANALYSIS_YEARS),
        ["jalali_year", "jalali_month"] + analysis_assets,
    ]
    .dropna(subset=analysis_assets)
    .copy()
)

FIXED_ASSET_COVARIANCE = (
    fixed_risk_sample[analysis_assets].cov() * 12
)
FIXED_EXCESS_COVARIANCE = (
    fixed_risk_sample[RISKY_ASSETS]
    .sub(fixed_risk_sample[FIXED_INCOME], axis=0)
    .cov()
    * 12
)

fixed_asset_volatility = (
    np.sqrt(np.diag(FIXED_ASSET_COVARIANCE))
    * 100
)
fixed_volatility_table = pd.DataFrame(
    {
        "asset": [ASSET_LABELS[asset] for asset in analysis_assets],
        "fixed_annualized_volatility_pct": fixed_asset_volatility,
    },
    index=analysis_assets,
)

assert len(fixed_risk_sample) == sum(EXPECTED_MONTHS.values())
assert np.linalg.eigvalsh(FIXED_ASSET_COVARIANCE).min() >= -1e-12
assert np.linalg.eigvalsh(FIXED_EXCESS_COVARIANCE).min() >= -1e-12

display(fixed_volatility_table.round(2))
display(FIXED_ASSET_COVARIANCE.round(6))

,asset,fixed_annualized_volatility_pct
gold_18k,Gold,35.0300
equity_tedpix,Equity,40.7600
tehran_housing,Housing,15.0400
fixed_income_etemad,Fixed Income,2.7400


asset_id,gold_18k,equity_tedpix,tehran_housing,fixed_income_etemad
asset_id,,,,
gold_18k,0.1227,0.0285,0.0109,0.0011
equity_tedpix,0.0285,0.1662,0.0162,0.0012
tehran_housing,0.0109,0.0162,0.0226,0.0003
fixed_income_etemad,0.0011,0.0012,0.0003,0.0007


## Alternative Stage I: yearly return with fixed full-sample tracking error

The numerator remains year-specific and is calculated from the realized buy-and-hold portfolio path. The denominator is now the fixed full-sample annualized tracking error. For year $y$,

$$S_y^{fixed}(w)=\frac{12\,\overline{d_y(w)}}{\sqrt{w^\top\Sigma_{excess}^{fixed}w}},$$

subject to long-only weights that sum to one. Thus identical weights always receive identical risk, regardless of year; only their realized yearly return contribution changes.

In [30]:
def fixed_tracking_error(weights: np.ndarray) -> float:
    weights = np.asarray(weights, dtype=float)
    variance = weights @ FIXED_EXCESS_COVARIANCE.to_numpy() @ weights
    return float(np.sqrt(max(variance, 0.0)))


def fixed_risk_stage_one_score(
    weights: np.ndarray,
    risky_returns: np.ndarray,
    benchmark_returns: np.ndarray,
) -> float:
    portfolio_returns, _ = simulate_buy_and_hold(weights, risky_returns)
    annualized_mean_differential = 12 * (
        portfolio_returns - benchmark_returns
    ).mean()
    tracking_error = fixed_tracking_error(weights)
    if tracking_error <= 1e-12:
        return -np.inf
    return annualized_mean_differential / tracking_error


def optimize_fixed_risk_stage_one(year_df, risky_assets, benchmark_asset, seed):
    risky_returns = year_df[risky_assets].to_numpy(float)
    benchmark_returns = year_df[benchmark_asset].to_numpy(float)
    rng = np.random.default_rng(seed)
    n_assets = len(risky_assets)
    starts = [
        np.full(n_assets, 1 / n_assets),
        *np.eye(n_assets),
        *rng.dirichlet(np.ones(n_assets), size=30),
    ]
    best = None
    for start in starts:
        result = minimize(
            lambda weights: -fixed_risk_stage_one_score(
                weights, risky_returns, benchmark_returns
            )
            if np.isfinite(
                fixed_risk_stage_one_score(weights, risky_returns, benchmark_returns)
            )
            else 1e12,
            x0=start,
            method="SLSQP",
            bounds=[(0.0, 1.0)] * n_assets,
            constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1},
            options={"maxiter": 2000, "ftol": 1e-12},
        )
        if result.success and (best is None or result.fun < best.fun):
            best = result
    if best is None:
        raise RuntimeError("No fixed-risk Stage I optimization succeeded.")
    return best


fixed_stage_one_rows = []
for year in ANALYSIS_YEARS:
    year_df = get_year_data(monthly, year, RISKY_ASSETS, FIXED_INCOME)
    result = optimize_fixed_risk_stage_one(
        year_df, RISKY_ASSETS, FIXED_INCOME, seed=year
    )
    portfolio_returns, ending_weights = simulate_buy_and_hold(
        result.x, year_df[RISKY_ASSETS].to_numpy(float)
    )
    benchmark_returns = year_df[FIXED_INCOME].to_numpy(float)
    row = {
        "year": year,
        "fixed_risk_score": -result.fun,
        "risky_sleeve_return": np.prod(1 + portfolio_returns) - 1,
        "benchmark_return": np.prod(1 + benchmark_returns) - 1,
        "fixed_tracking_error": fixed_tracking_error(result.x),
    }
    for asset, initial_weight, ending_weight in zip(
        RISKY_ASSETS, result.x, ending_weights
    ):
        row[f"initial_weight_{asset}"] = initial_weight
        row[f"ending_weight_{asset}"] = ending_weight
    fixed_stage_one_rows.append(row)

fixed_stage_one_results = pd.DataFrame(fixed_stage_one_rows)
assert np.allclose(
    fixed_stage_one_results[[f"initial_weight_{a}" for a in RISKY_ASSETS]].sum(axis=1),
    1.0,
)

fixed_stage_one_display = fixed_stage_one_results.copy()
fixed_stage_one_pct_columns = [
    column for column in fixed_stage_one_display.columns
    if column.startswith(("initial_weight_", "ending_weight_"))
    or column in {"risky_sleeve_return", "benchmark_return", "fixed_tracking_error"}
]
fixed_stage_one_display[fixed_stage_one_pct_columns] *= 100
display(fixed_stage_one_display.round(2))

,year,fixed_risk_score,risky_sleeve_return,benchmark_return,fixed_tracking_error,initial_weight_gold_18k,ending_weight_gold_18k,initial_weight_equity_tedpix,ending_weight_equity_tedpix,initial_weight_tehran_housing,ending_weight_tehran_housing
0,1396,0.2400,29.5600,23.9100,20.4600,50.8300,52.1400,0.0000,0.0000,49.1700,47.8600
1,1397,3.8900,116.5800,22.2600,15.9300,28.0700,36.3300,0.1300,0.1100,71.8000,63.5600
2,1398,2.5200,110.0000,22.8500,22.4600,0.4900,0.3300,47.0400,64.3100,52.4700,35.3600
3,1399,3.6400,103.9600,21.5600,15.6800,5.6500,4.9900,17.9800,22.4800,76.3700,72.5300
4,1400,-0.1400,13.6000,20.9800,34.8100,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000
5,1401,3.1100,90.4000,23.4200,14.7900,16.3200,18.3600,0.0000,0.0000,83.6800,81.6400
6,1402,-0.0100,23.2100,26.1700,34.8100,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000
7,1403,2.0700,146.6600,31.5600,34.2000,98.0800,99.1000,0.0000,0.0000,1.9200,0.9000
8,1404,1.7000,117.5200,34.9500,34.8100,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000
9,1405,6.4100,66.0200,14.8900,14.9600,0.0000,0.0000,3.9400,3.8700,96.0600,96.1300


In [31]:
fixed_weights_long = (
    fixed_stage_one_results[
        ["year"] + [f"initial_weight_{asset}" for asset in RISKY_ASSETS]
    ]
    .melt(id_vars="year", var_name="asset", value_name="weight")
)
fixed_weights_long["asset"] = (
    fixed_weights_long["asset"]
    .str.replace("initial_weight_", "", regex=False)
    .map(ASSET_LABELS)
)
fixed_weights_long["weight_pct"] = fixed_weights_long["weight"] * 100

fig = px.bar(
    fixed_weights_long,
    x="year",
    y="weight_pct",
    color="asset",
    labels={
        "year": "Solar Hijri Year",
        "weight_pct": "Initial risky-sleeve weight (%)",
        "asset": "Asset",
    },
    title="Alternative Stage I Allocation Using Fixed Full-Sample Risk",
)
fig.update_layout(barmode="stack", yaxis_range=[0, 100])
fig.show()

## Alternative Stage II: fixed full-sample portfolio volatility

Stage II again varies only the total risky share $\alpha$. Realized compounded return is calculated from each year's buy-and-hold path, while volatility is calculated from the one fixed covariance matrix:

$$\sigma_{total}^{fixed}(\alpha)=\sqrt{x(\alpha)^\top\Sigma^{fixed}x(\alpha)},$$

where $x(\alpha)=(\alpha w_y^*,1-\alpha)$. The utility grid and gamma values are the same as in the original Stage II section.

In [32]:
def fixed_total_portfolio_volatility(risky_weights, alpha):
    total_weights = np.concatenate([alpha * risky_weights, [1 - alpha]])
    variance = (
        total_weights
        @ FIXED_ASSET_COVARIANCE.loc[analysis_assets, analysis_assets].to_numpy()
        @ total_weights
    )
    return float(np.sqrt(max(variance, 0.0)))


def run_fixed_risk_stage_two(
    monthly_df, fixed_stage_one_results, gamma_values, grid_size=5001
):
    rows = []
    alpha_grid = np.linspace(0.0, 1.0, grid_size)
    weight_columns = [f"initial_weight_{asset}" for asset in RISKY_ASSETS]
    for _, stage_one_row in fixed_stage_one_results.iterrows():
        year = int(stage_one_row["year"])
        year_df = get_year_data(monthly_df, year, RISKY_ASSETS, FIXED_INCOME)
        risky_weights = stage_one_row[weight_columns].to_numpy(float)
        risky_growth = np.prod(1 + year_df[RISKY_ASSETS].to_numpy(float), axis=0)
        risky_ending_wealth = float(risky_weights @ risky_growth)
        fixed_ending_wealth = float(
            np.prod(1 + year_df[FIXED_INCOME].to_numpy(float))
        )
        annual_returns = (
            alpha_grid * risky_ending_wealth
            + (1 - alpha_grid) * fixed_ending_wealth
            - 1
        )
        total_weight_grid = np.column_stack(
            [alpha_grid[:, None] * risky_weights, 1 - alpha_grid]
        )
        covariance = FIXED_ASSET_COVARIANCE.loc[
            analysis_assets, analysis_assets
        ].to_numpy()
        fixed_variances = np.einsum(
            "ij,jk,ik->i", total_weight_grid, covariance, total_weight_grid
        )
        fixed_volatilities = np.sqrt(np.maximum(fixed_variances, 0.0))
        for gamma in gamma_values:
            utilities = annual_returns - 0.5 * gamma * fixed_volatilities**2
            best_index = int(np.argmax(utilities))
            best_alpha = alpha_grid[best_index]
            row = {
                "year": year,
                "gamma": gamma,
                "alpha": best_alpha,
                "fixed_income_weight": 1 - best_alpha,
                "annual_return": annual_returns[best_index],
                "fixed_annualized_volatility": fixed_volatilities[best_index],
                "utility": utilities[best_index],
            }
            for asset, risky_weight in zip(RISKY_ASSETS, risky_weights):
                row[f"total_weight_{asset}"] = best_alpha * risky_weight
            rows.append(row)
    return pd.DataFrame(rows)


fixed_stage_two_results = run_fixed_risk_stage_two(
    monthly, fixed_stage_one_results, GAMMA_VALUES
)
fixed_alpha_table = (
    fixed_stage_two_results.pivot(index="year", columns="gamma", values="alpha")
    * 100
)
display(fixed_alpha_table.round(2))

fixed_gamma_4 = fixed_stage_two_results.loc[
    fixed_stage_two_results["gamma"].eq(REPRESENTATIVE_GAMMA)
].copy()
fixed_gamma_4_pct_columns = [
    "alpha", "fixed_income_weight", "annual_return",
    "fixed_annualized_volatility",
] + [f"total_weight_{asset}" for asset in RISKY_ASSETS]
fixed_gamma_4[fixed_gamma_4_pct_columns] *= 100
display(fixed_gamma_4.round(2))

gamma,0,1,2,4,6,8,10,15,20,25,30,35,40,45,50
year,,,,,,,,,,,,,,,
1396,100.0000,100.0000,67.5200,33.7600,22.5200,16.8800,13.5200,9.0200,6.7600,5.4200,4.5200,3.8800,3.3800,3.0200,2.7200
1397,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,93.6400,83.3200,75.0600
1398,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,86.3800,69.1000,57.5800,49.3600,43.1800,38.3800,34.5400
1399,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,96.5600,84.6000,75.2800,67.8400
1400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1401,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,88.7200,77.7800,69.2800,62.4800
1402,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1403,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,98.1200,65.3000,48.9000,39.0600,32.5000,27.8000,24.3000,21.5600,19.3800
1404,100.0000,100.0000,100.0000,100.0000,100.0000,84.8600,67.8200,45.1200,33.7600,26.9400,22.4000,19.1600,16.7200,14.8200,13.3200


,year,gamma,alpha,fixed_income_weight,annual_return,fixed_annualized_volatility,utility,total_weight_gold_18k,total_weight_equity_tedpix,total_weight_tehran_housing
3,1396,4,33.7600,66.2400,25.8200,7.4300,0.2500,17.1600,0.0000,16.6000
18,1397,4,100.0000,0.0000,116.5800,16.0500,1.1100,28.0700,0.1300,71.8000
33,1398,4,100.0000,0.0000,110.0000,22.6300,1.0000,0.4900,47.0400,52.4700
48,1399,4,100.0000,0.0000,103.9600,15.7900,0.9900,5.6500,17.9800,76.3700
63,1400,4,0.0000,100.0000,20.9800,2.7400,0.2100,0.0000,0.0000,0.0000
78,1401,4,100.0000,0.0000,90.4000,14.8600,0.8600,16.3200,0.0000,83.6800
93,1402,4,0.0000,100.0000,26.1700,2.7400,0.2600,0.0000,0.0000,0.0000
108,1403,4,100.0000,0.0000,146.6600,34.4100,1.2300,98.0800,0.0000,1.9200
123,1404,4,100.0000,0.0000,117.5200,35.0300,0.9300,100.0000,0.0000,0.0000
138,1405,4,100.0000,0.0000,66.0200,14.9600,0.6200,0.0000,3.9400,96.0600


In [33]:
fig = px.imshow(
    fixed_alpha_table,
    text_auto=".1f",
    aspect="auto",
    labels={
        "x": "Risk Aversion (γ)",
        "y": "Solar Hijri Year",
        "color": "Risky Allocation (%)",
    },
    title="Alternative Stage II: Risky Allocation with Fixed Full-Sample Volatility",
)
fig.show()

## How to interpret the alternative results

The fixed annualized volatilities are **35.03% for gold, 40.76% for equity, 15.04% for housing, and 2.74% for fixed income**. These values are estimated once and do not change between yearly rows.

The alternative Stage I results are generally more diversified than the original year-specific-risk results. For example, 1396 changes from 84.37% gold / 15.63% housing to **50.83% gold / 49.17% housing**. In 1398, the original 100% equity result becomes **47.04% equity / 52.47% housing**. In 1405 YTD, housing rises to **96.06%**, with 3.94% equity and no gold. However, 1400, 1402, and 1404 remain 100% gold, while 1403 remains 98.08% gold. Concentration therefore declines in several years but does not disappear. Changing realized returns remain a major driver.

Stage II becomes more responsive to risk aversion because the same full-sample covariance scale is applied consistently. At the illustrative $\gamma=4$, 1396 allocates **33.76% to the risky sleeve and 66.24% to fixed income**, whereas the original specification selected 100% risky exposure. The weak-risky years 1400 and 1402 still select 100% fixed income. The other years remain fully risky at $\gamma=4$ because their realized return advantages are large. At higher gamma values, exposure declines smoothly in more years; for example, at $\gamma=50$ the risky shares are 2.72% in 1396, 34.54% in 1398, 19.38% in 1403, 13.32% in 1404, and 47.40% in 1405 YTD.

**Conclusion.** Fixing volatility across years removes one source of instability—noisy risk estimates based on only 12 observations—and produces more diversified allocations in several periods. It does not produce a stable universal portfolio because annual realized returns still change sharply.

**Important qualification.** The fixed covariance matrix uses the entire 1396–1405/05 sample. Earlier-year decisions therefore use information from later months. This is appropriate for the requested descriptive comparison, but an investable implementation would estimate volatility only from information available before each decision date, normally with a rolling or expanding window.